In [1]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.models import load_model

I0000 00:00:1790782682.409476   23064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790782683.745916   23064 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790782688.655216   23064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
model = load_model("model.keras")

E0000 00:00:1790782692.902843   23064 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [4]:
df = pd.read_csv("Dataset/hour.csv")

In [5]:
df["datetime"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")

In [6]:
df = df.sort_values("datetime")
df = df.set_index("datetime")

In [7]:
df = df[["cnt"]]

In [8]:
SEQUENCE_LENGTH = 24
latest_data = df["cnt"].values[-SEQUENCE_LENGTH:]
print("Latest observations:", len(latest_data))

Latest observations: 24


In [9]:
latest_scaled = scaler.transform(latest_data.reshape(-1, 1))

In [10]:
X_input = latest_scaled.reshape(1, SEQUENCE_LENGTH, 1)
print("Input Shape:", X_input.shape)

Input Shape: (1, 24, 1)


In [11]:
prediction_scaled = model.predict(X_input)
prediction = scaler.inverse_transform(prediction_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 812ms/step


In [12]:
predicted_bikes = prediction[0][0]
print(f"Predicted next-hour bike demand: " f"{predicted_bikes:.0f} bikes")

Predicted next-hour bike demand: 50 bikes


In [13]:
print("Latest 10 observations:")
print(df.tail(10))

Latest 10 observations:
                     cnt
datetime                
2012-12-31 14:00:00  247
2012-12-31 15:00:00  315
2012-12-31 16:00:00  214
2012-12-31 17:00:00  164
2012-12-31 18:00:00  122
2012-12-31 19:00:00  119
2012-12-31 20:00:00   89
2012-12-31 21:00:00   90
2012-12-31 22:00:00   61
2012-12-31 23:00:00   49
